# 09 - Silver: Annual Macro Fact Table

This notebook creates `silver.fact_macro_annual`, one row per project country per year from 1990 to 2024.

Inputs:

- `silver.dim_country`
- `silver.dim_bloc_membership`
- `bronze.data360_raw`
- `bronze.imf_weo_raw`

World Bank Data360 is the primary source for GDP, GDP per capita, population, and headline CPI inflation. IMF WEO provides fiscal and external context: debt-to-GDP, revenue, expenditure, net lending/borrowing, current account balance, real GDP growth, and a second inflation series.

In [ ]:
# Cell 1 - Configuration
from datetime import datetime, timezone

from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

START_YEAR = 1990
END_YEAR = 2024
loaded_at = datetime.now(timezone.utc)

DATA360_INDICATORS = {
    "NY.GDP.MKTP.CD": "gdp_current_usd_wb",
    "NY.GDP.PCAP.CD": "gdp_per_capita_current_usd_wb",
    "SP.POP.TOTL": "population_wb",
    "FP.CPI.TOTL.ZG": "inflation_cpi_pct_wb",
}

WEO_INDICATORS = {
    "GGXWDG_NGDP": "gross_debt_pct_gdp_imf",
    "GGR_NGDP": "government_revenue_pct_gdp_imf",
    "GGX_NGDP": "government_expenditure_pct_gdp_imf",
    "GGXCNL_NGDP": "net_lending_borrowing_pct_gdp_imf",
    "BCA_NGDPD": "current_account_balance_pct_gdp_imf",
    "NGDP_RPCH": "real_gdp_growth_pct_imf",
    "PCPIPCH": "inflation_cpi_pct_imf",
    "NGDPD": "gdp_current_usd_weo_raw",
}

print("Target table: silver.fact_macro_annual")
print(f"Years: {START_YEAR}-{END_YEAR}")

In [ ]:
# Cell 2 - Input table checks
def table_exists(schema_name, table_name):
    return spark.sql(f"SHOW TABLES IN {schema_name} LIKE '{table_name}'").count() > 0


required_tables = [
    ("silver", "dim_country"),
    ("silver", "dim_bloc_membership"),
    ("bronze", "data360_raw"),
    ("bronze", "imf_weo_raw"),
]

missing_tables = [f"{schema}.{table}" for schema, table in required_tables if not table_exists(schema, table)]
if missing_tables:
    raise ValueError(f"Missing required input table(s): {missing_tables}")

print("All required tables found.")

print("Data360 input coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT country_iso3) AS countries,
        COUNT(DISTINCT indicator_code) AS indicators,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        COUNT(value) AS non_null_values
    FROM bronze.data360_raw
""").show()

print("IMF WEO input coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT country_iso3) AS countries,
        COUNT(DISTINCT indicator_code) AS indicators,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        COUNT(value) AS non_null_values
    FROM bronze.imf_weo_raw
""").show()

In [ ]:
# Cell 3 - Build country-year panel with time-aware analytical bloc
country_dim = spark.table("silver.dim_country").select(
    "country_key",
    "country_iso3",
    "country_iso2",
    "m49_code",
    "country_name",
    "official_name",
    "region",
)

years_df = spark.range(START_YEAR, END_YEAR + 1).select(F.col("id").cast("int").alias("year"))

panel_df = (
    country_dim.crossJoin(years_df)
    .withColumn("year_end_date", F.expr("make_date(year, 12, 31)"))
)

bloc_df = spark.table("silver.dim_bloc_membership").where("is_primary_analytical_bloc = true").select(
    F.col("country_iso3").alias("bloc_country_iso3"),
    "bloc_code",
    "bloc_name",
    "valid_from",
    "valid_to",
)

panel_with_bloc_df = (
    panel_df.alias("p")
    .join(
        bloc_df.alias("b"),
        (F.col("p.country_iso3") == F.col("b.bloc_country_iso3"))
        & (F.col("p.year_end_date") >= F.col("b.valid_from"))
        & (F.col("b.valid_to").isNull() | (F.col("p.year_end_date") <= F.col("b.valid_to"))),
        "left",
    )
    .select(
        F.col("p.country_key"),
        F.col("p.country_iso3"),
        F.col("p.country_iso2"),
        F.col("p.m49_code"),
        F.col("p.country_name"),
        F.col("p.official_name"),
        F.col("p.region"),
        F.col("p.year"),
        F.col("p.year_end_date"),
        F.col("b.bloc_code").alias("analytical_bloc_code"),
        F.col("b.bloc_name").alias("analytical_bloc_name"),
    )
)

missing_bloc_rows = panel_with_bloc_df.where(F.col("analytical_bloc_code").isNull()).count()
if missing_bloc_rows:
    raise ValueError(f"Country-year rows missing analytical bloc: {missing_bloc_rows}")

print(f"Country-year panel rows: {panel_with_bloc_df.count():,}")
panel_with_bloc_df.groupBy("analytical_bloc_code").count().orderBy("analytical_bloc_code").show()

In [ ]:
# Cell 4 - Pivot World Bank Data360 indicators
data360_raw = (
    spark.table("bronze.data360_raw")
    .where((F.col("year") >= START_YEAR) & (F.col("year") <= END_YEAR))
    .where(F.col("indicator_code").isin(list(DATA360_INDICATORS.keys())))
)

data360_agg_exprs = [
    F.max(F.when(F.col("indicator_code") == indicator_code, F.col("value"))).alias(column_name)
    for indicator_code, column_name in DATA360_INDICATORS.items()
]

data360_macro_df = data360_raw.groupBy("country_iso3", "year").agg(*data360_agg_exprs)

print(f"Data360 macro rows: {data360_macro_df.count():,}")
data360_macro_df.show(10, truncate=False)

In [ ]:
# Cell 5 - Pivot IMF WEO indicators
weo_raw = (
    spark.table("bronze.imf_weo_raw")
    .where((F.col("year") >= START_YEAR) & (F.col("year") <= END_YEAR))
    .where(F.col("indicator_code").isin(list(WEO_INDICATORS.keys())))
)

weo_agg_exprs = [
    F.max(F.when(F.col("indicator_code") == indicator_code, F.col("value"))).alias(column_name)
    for indicator_code, column_name in WEO_INDICATORS.items()
]

weo_macro_df = weo_raw.groupBy("country_iso3", "year").agg(*weo_agg_exprs)

print(f"IMF WEO macro rows: {weo_macro_df.count():,}")
weo_macro_df.show(10, truncate=False)

In [ ]:
# Cell 6 - Combine sources and derive analysis-ready measures
fact_df = (
    panel_with_bloc_df.alias("p")
    .join(data360_macro_df.alias("d"), ["country_iso3", "year"], "left")
    .join(weo_macro_df.alias("w"), ["country_iso3", "year"], "left")
    .select(
        F.col("p.country_key"),
        F.col("country_iso3"),
        F.col("p.country_iso2"),
        F.col("p.m49_code"),
        F.col("p.country_name"),
        F.col("p.region"),
        F.col("p.analytical_bloc_code"),
        F.col("p.analytical_bloc_name"),
        F.col("year"),
        F.col("p.year_end_date"),
        F.col("d.gdp_current_usd_wb"),
        F.col("d.gdp_per_capita_current_usd_wb"),
        F.col("d.population_wb"),
        F.col("d.inflation_cpi_pct_wb"),
        F.col("w.real_gdp_growth_pct_imf"),
        F.col("w.inflation_cpi_pct_imf"),
        F.col("w.gross_debt_pct_gdp_imf"),
        F.col("w.government_revenue_pct_gdp_imf"),
        F.col("w.government_expenditure_pct_gdp_imf"),
        F.col("w.net_lending_borrowing_pct_gdp_imf"),
        F.col("w.current_account_balance_pct_gdp_imf"),
        F.col("w.gdp_current_usd_weo_raw"),
    )
)

fact_df = (
    fact_df
    .withColumn("gdp_current_usd", F.col("gdp_current_usd_wb"))
    .withColumn(
        "gdp_per_capita_current_usd",
        F.coalesce(
            F.col("gdp_per_capita_current_usd_wb"),
            F.when(F.col("population_wb") > 0, F.col("gdp_current_usd_wb") / F.col("population_wb")),
        ),
    )
    .withColumn("population", F.col("population_wb"))
    .withColumn("inflation_cpi_pct", F.coalesce(F.col("inflation_cpi_pct_wb"), F.col("inflation_cpi_pct_imf")))
    .withColumn("gdp_current_usd_billions", F.col("gdp_current_usd") / F.lit(1_000_000_000.0))
    .withColumn("population_millions", F.col("population") / F.lit(1_000_000.0))
    .withColumn("gross_debt_usd", F.col("gdp_current_usd") * F.col("gross_debt_pct_gdp_imf") / F.lit(100.0))
    .withColumn("government_revenue_usd", F.col("gdp_current_usd") * F.col("government_revenue_pct_gdp_imf") / F.lit(100.0))
    .withColumn("government_expenditure_usd", F.col("gdp_current_usd") * F.col("government_expenditure_pct_gdp_imf") / F.lit(100.0))
    .withColumn("net_lending_borrowing_usd", F.col("gdp_current_usd") * F.col("net_lending_borrowing_pct_gdp_imf") / F.lit(100.0))
    .withColumn("current_account_balance_usd", F.col("gdp_current_usd") * F.col("current_account_balance_pct_gdp_imf") / F.lit(100.0))
    .withColumn("has_world_bank_macro", F.col("gdp_current_usd").isNotNull() & F.col("population").isNotNull())
    .withColumn("has_imf_fiscal_context", F.col("gross_debt_pct_gdp_imf").isNotNull())
    .withColumn(
        "missing_core_metric_count",
        F.expr("""
            CASE WHEN gdp_current_usd IS NULL THEN 1 ELSE 0 END +
            CASE WHEN gdp_per_capita_current_usd IS NULL THEN 1 ELSE 0 END +
            CASE WHEN population IS NULL THEN 1 ELSE 0 END +
            CASE WHEN inflation_cpi_pct IS NULL THEN 1 ELSE 0 END +
            CASE WHEN gross_debt_pct_gdp_imf IS NULL THEN 1 ELSE 0 END +
            CASE WHEN real_gdp_growth_pct_imf IS NULL THEN 1 ELSE 0 END
        """),
    )
    .withColumn("loaded_at", F.lit(loaded_at))
)

ordered_columns = [
    "country_key",
    "country_iso3",
    "country_iso2",
    "m49_code",
    "country_name",
    "region",
    "analytical_bloc_code",
    "analytical_bloc_name",
    "year",
    "year_end_date",
    "gdp_current_usd",
    "gdp_current_usd_billions",
    "gdp_per_capita_current_usd",
    "population",
    "population_millions",
    "inflation_cpi_pct",
    "real_gdp_growth_pct_imf",
    "gross_debt_pct_gdp_imf",
    "gross_debt_usd",
    "government_revenue_pct_gdp_imf",
    "government_revenue_usd",
    "government_expenditure_pct_gdp_imf",
    "government_expenditure_usd",
    "net_lending_borrowing_pct_gdp_imf",
    "net_lending_borrowing_usd",
    "current_account_balance_pct_gdp_imf",
    "current_account_balance_usd",
    "gdp_current_usd_wb",
    "gdp_per_capita_current_usd_wb",
    "population_wb",
    "inflation_cpi_pct_wb",
    "inflation_cpi_pct_imf",
    "gdp_current_usd_weo_raw",
    "has_world_bank_macro",
    "has_imf_fiscal_context",
    "missing_core_metric_count",
    "loaded_at",
]

fact_df = fact_df.select(*ordered_columns)

print(f"Fact rows: {fact_df.count():,}")
fact_df.printSchema()
fact_df.show(10, truncate=False)

In [ ]:
# Cell 7 - Write silver.fact_macro_annual
spark.sql("DROP TABLE IF EXISTS silver.fact_macro_annual")

(fact_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fact_macro_annual"))

print("Write complete: silver.fact_macro_annual")

In [ ]:
# Cell 8 - Verification and sanity checks
print("Overall coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        SUM(CASE WHEN has_world_bank_macro THEN 1 ELSE 0 END) AS rows_with_wb_macro,
        SUM(CASE WHEN has_imf_fiscal_context THEN 1 ELSE 0 END) AS rows_with_imf_fiscal,
        ROUND(AVG(missing_core_metric_count), 2) AS avg_missing_core_metrics
    FROM silver.fact_macro_annual
""").show()

print("Coverage by analytical bloc and year range:")
spark.sql("""
    SELECT
        analytical_bloc_code,
        COUNT(*) AS rows,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        SUM(CASE WHEN gdp_current_usd IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_gdp,
        SUM(CASE WHEN gross_debt_pct_gdp_imf IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_debt
    FROM silver.fact_macro_annual
    GROUP BY analytical_bloc_code
    ORDER BY analytical_bloc_code
""").show(truncate=False)

print("Latest macro snapshot:")
spark.sql("""
    SELECT
        analytical_bloc_code,
        country_iso3,
        country_name,
        ROUND(gdp_current_usd_billions, 2) AS gdp_billions_usd,
        ROUND(population_millions, 1) AS population_millions,
        ROUND(gdp_per_capita_current_usd, 0) AS gdp_per_capita_usd,
        ROUND(gross_debt_pct_gdp_imf, 1) AS debt_pct_gdp,
        ROUND(net_lending_borrowing_pct_gdp_imf, 1) AS fiscal_balance_pct_gdp
    FROM silver.fact_macro_annual
    WHERE year = (SELECT MAX(year) FROM silver.fact_macro_annual)
    ORDER BY gdp_current_usd DESC
""").show(25, truncate=False)

print("Bloc aggregates, latest year:")
spark.sql("""
    SELECT
        analytical_bloc_code,
        COUNT(*) AS countries,
        ROUND(SUM(gdp_current_usd) / 1e9, 2) AS bloc_gdp_billions_usd,
        ROUND(SUM(population) / 1e6, 1) AS bloc_population_millions,
        ROUND(SUM(gdp_current_usd) / SUM(population), 0) AS bloc_weighted_gdp_per_capita_usd,
        ROUND(SUM(gross_debt_usd) / SUM(gdp_current_usd) * 100, 1) AS bloc_weighted_debt_pct_gdp
    FROM silver.fact_macro_annual
    WHERE year = (SELECT MAX(year) FROM silver.fact_macro_annual)
    GROUP BY analytical_bloc_code
    ORDER BY analytical_bloc_code
""").show(truncate=False)

In [ ]:
# Cell 9 - Data quality checks for null-heavy rows
print("Countries with weakest macro completeness:")
spark.sql("""
    SELECT
        country_iso3,
        country_name,
        ROUND(AVG(missing_core_metric_count), 2) AS avg_missing_core_metrics,
        SUM(CASE WHEN gdp_current_usd IS NULL THEN 1 ELSE 0 END) AS missing_gdp_years,
        SUM(CASE WHEN gross_debt_pct_gdp_imf IS NULL THEN 1 ELSE 0 END) AS missing_debt_years,
        SUM(CASE WHEN inflation_cpi_pct IS NULL THEN 1 ELSE 0 END) AS missing_inflation_years
    FROM silver.fact_macro_annual
    GROUP BY country_iso3, country_name
    ORDER BY avg_missing_core_metrics DESC, country_iso3
""").show(25, truncate=False)

print("Cameroon macro trajectory, selected years:")
spark.sql("""
    SELECT
        year,
        ROUND(gdp_current_usd_billions, 2) AS gdp_billions_usd,
        ROUND(population_millions, 1) AS population_millions,
        ROUND(gdp_per_capita_current_usd, 0) AS gdp_per_capita_usd,
        ROUND(real_gdp_growth_pct_imf, 1) AS real_gdp_growth_pct,
        ROUND(inflation_cpi_pct, 1) AS inflation_pct,
        ROUND(gross_debt_pct_gdp_imf, 1) AS debt_pct_gdp
    FROM silver.fact_macro_annual
    WHERE country_iso3 = 'CMR'
      AND year IN (1990, 2000, 2010, 2020, 2024)
    ORDER BY year
""").show(truncate=False)